# 02 — Preprocesado clásico: CLAHE en profundidad

Este notebook se centra **solo en CLAHE**, la pieza más importante del preprocesado, para que entiendas cómo afecta `clip_limit` al resultado.

Para ver el pipeline completo (White Balance + CLAHE + Denoising), pasa al `02b_preprocesado_completo.ipynb`.

**Pre-requisitos:** dataset combinado en `data/external/combined/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import apply_clahe, preprocess
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/combined')
plt.rcParams['figure.dpi'] = 90

## 1. Comprobar el dataset y elegir una imagen

In [ ]:
if not DATA_ROOT.is_dir():
    print(f'❌  No existe {DATA_ROOT}')
    print('Ejecuta: python scripts/combine_datasets.py')
else:
    categories = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
    print(f'✓  Encontradas {len(categories)} clases originales en el dataset combinado.')
    print(f'   Algunas: {", ".join(categories[:8])}...')

## 2. Antes / después en una sola imagen

Cogemos una imagen y le aplicamos CLAHE con los parámetros por defecto. Si quieres cambiar de imagen, edita la variable `category` (cualquiera del listado de la celda anterior).

In [ ]:
# Cambia esta categoría para probar con distintos productos
category = 'Apple'  # otras opciones: 'CEREAL', 'JUICE', 'CHIPS', 'JAM'...

cat_dir = DATA_ROOT / category
if not cat_dir.is_dir():
    print(f'⚠  No existe {cat_dir}, prueba otra categoría del listado anterior.')
else:
    img = load_image(list_images(cat_dir)[0])
    img_clahe = apply_clahe(img)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(img_clahe); axes[1].set_title('CLAHE (clip=2.0, grid=8x8)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

## 3. Comparativa de `clip_limit`

Probamos cómo cambia el resultado al variar el parámetro de contraste. Más alto significa más contraste local pero también amplifica más el ruido.

In [ ]:
clip_limits = [1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, len(clip_limits) + 1,
                         figsize=(4 * (len(clip_limits) + 1), 4))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')

for ax, cl in zip(axes[1:], clip_limits):
    out = apply_clahe(img, clip_limit=cl)
    ax.imshow(out); ax.set_title(f'clip_limit = {cl}'); ax.axis('off')

plt.tight_layout(); plt.show()
print('→ Observa cómo a partir de clip_limit=4 los colores empiezan a saturarse')
print('  artificialmente. El sweet spot suele estar en 2.0-3.0.')

## 4. Histograma del canal L (luminosidad)

CLAHE redistribuye los valores del canal L para usar mejor el rango disponible. El histograma del original suele estar concentrado; el procesado se ensancha.

In [ ]:
def get_l_channel(rgb_img):
    bgr = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    return lab[:, :, 0]

l_orig = get_l_channel(img)
l_clahe = get_l_channel(img_clahe)

fig, ax = plt.subplots(1, 1, figsize=(9, 4))
ax.hist(l_orig.ravel(), bins=50, alpha=0.6, label='Original', color='#0A0A0A')
ax.hist(l_clahe.ravel(), bins=50, alpha=0.6, label='CLAHE', color='#FF6B00')
ax.set_xlabel('Valor de L (luminosidad)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma del canal L: antes vs después de CLAHE')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Galería sobre 6 categorías

Para hacerse una idea del efecto en distintos tipos de producto.

In [ ]:
sample_categories = ['Apple', 'CEREAL', 'JUICE', 'CHIPS', 'JAM', 'TEA']
samples = []
for c in sample_categories:
    cat_dir = DATA_ROOT / c
    if cat_dir.is_dir():
        imgs = list_images(cat_dir)
        if imgs:
            samples.append((c, imgs[0]))

fig, axes = plt.subplots(len(samples), 2, figsize=(8, 3 * len(samples)))
for i, (cat, path) in enumerate(samples):
    img = load_image(path)
    img_pre = apply_clahe(img)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f'{cat} — original', fontsize=10); axes[i, 0].axis('off')
    axes[i, 1].imshow(img_pre); axes[i, 1].set_title(f'{cat} — CLAHE', fontsize=10); axes[i, 1].axis('off')
plt.tight_layout(); plt.show()

## 6. Conclusiones

- **Colores conservados**: CLAHE sobre el canal L de Lab respeta los colores; solo iguala el contraste.
- **clip_limit = 2.0** es el equilibrio que usamos por defecto en el pipeline.
- **Cuidado con valores altos**: a partir de 4-8 se empiezan a notar artefactos.

**Próximo paso:** `02b_preprocesado_completo.ipynb` añade White Balance + Denoising al pipeline.